# 04 - Delivery Impact
**Questions:** how much do late deliveries hurt reviews, does a late first delivery reduce repeat purchasing, and what could improving on-time delivery be worth?

Sources: `marts.mart_delivery_impact`, `marts.mart_delivery_vs_repeat`, `marts.metrics_summary`.
**Everything here shows association, not proof of cause.** Late orders may differ in other ways (distance, seller, product).

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import get_warehouse_path  # noqa: E402

con = duckdb.connect(str(get_warehouse_path()), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    return con.execute(sql).df()

## 1. Review scores by lateness

In [ ]:
impact = q("SELECT * FROM marts.mart_delivery_impact ORDER BY bucket_order")
display(impact)
px.bar(impact, x="delivery_bucket", y="avg_review_score", text="avg_review_score",
       title="Average review score by delivery lateness").show()
px.bar(impact, x="delivery_bucket", y="pct_low_reviews", text="pct_low_reviews",
       title="% of reviews scoring 1-2 by delivery lateness").show()

## 2. Does a late first delivery reduce repeat purchasing?

In [ ]:
vr = q("SELECT * FROM marts.mart_delivery_vs_repeat")
display(vr)

Eligible customers have a first delivered order inside the window and at least the full follow-up horizon (see `repeat_horizon_days` in `dbt_project.yml`) before the window ends, so every customer is observed for the same length of time.
Check the customer counts: if the 'late' group is small, the repeat-rate gap is noisy.

## 3. Scenario: what could better on-time delivery be worth?

In [ ]:
# --- Assumptions you can change -------------------------------------------------
SHIFT_SHARES = [0.25, 0.50, 1.00]     # share of late first deliveries that become on-time
EFFECT_REALISED = [0.50, 1.00]        # how much of the observed repeat-rate gap is truly causal
# --------------------------------------------------------------------------------
summary = q("SELECT * FROM marts.metrics_summary").iloc[0]
vr = vr.set_index("first_delivery")

if not {"late", "on time or early"} <= set(vr.index):
    print("Need both groups to run the scenario.")
else:
    late, ontime = vr.loc["late"], vr.loc["on time or early"]
    gap = (ontime["repeat_rate_pct"] - late["repeat_rate_pct"]) / 100
    late_share = late["customers"] / (late["customers"] + ontime["customers"])
    print(f"Observed repeat-rate gap (on time minus late): {gap * 100:.2f} percentage points")
    print(f"Share of eligible customers whose first delivery was late: {late_share * 100:.1f}%")
    if gap <= 0:
        print("No positive gap observed, so this data does not support a revenue uplift from on-time delivery.")
    else:
        rows = []
        for shift in SHIFT_SHARES:
            for realised in EFFECT_REALISED:
                extra_repeaters = summary["customers"] * late_share * shift * gap * realised
                rows.append({
                    "late_first_orders_fixed": f"{shift:.0%}",
                    "gap_actually_causal": f"{realised:.0%}",
                    "extra_repeat_customers": extra_repeaters,
                    "extra_revenue_BRL": extra_repeaters * float(summary["aov"]),
                })
        display(pd.DataFrame(rows))
        print("Assumes each extra repeat customer places one more order at the average order value,")
        print("and that the eligible group is representative of all customers in the window.")

## How to read this
- Report the scenario as a **range with stated assumptions**, never a single confident number.
- If the repeat-rate gap is small or negative, say so: on-time delivery may matter more for reviews and reputation than for repeat revenue.
- Put the headline review-score gap and the scenario range in `docs/insights.md`.